# FloodShift-V3 — Conservative RIFT High-Standard Run

This run is a **validation-only recovery and improvement** after V2 failed its promotion gate.

## What V2 taught us

V2 improved temporary-flood validation IoU from the frozen V1 reference (~0.321) to **0.346**, but permanent-water IoU collapsed to **0.111**, pulling validation mIoU down to **0.477**.

Therefore V3 does **not** add more complexity indiscriminately.

## V3 principle

**Preserve the strong V1 representation first; refine it conservatively.**

Changes:

- restore V1 inverse-frequency class weights;
- no SAR modality dropout;
- no S4 RIFT;
- S3 RIFT starts as an exact identity residual;
- ASPP also starts as an exact identity residual;
- freeze the V1 backbone for the first 2 epochs;
- unfreeze with very low base learning rates;
- apply asymmetric focal-Tversky only to temporary flood;
- use only low-weight auxiliary/boundary losses;
- retain EMA and validation-only checkpoint selection.

## Required Kaggle inputs

Attach:

1. `01_FloodShift_Data_Audit`
2. `03_FloodShift_Preprocessing_FINAL_COMPLETE`
3. downloaded `FloodShift_Raw_Main_best.pt`

## Promotion gate

The final test remains locked until V3 achieves ALL of:

- validation mIoU ≥ **0.540**
- temporary-flood IoU ≥ **0.340**
- permanent-water IoU ≥ **0.230**

The ambitious final-test targets remain:

- strong: **mIoU ≥ 0.72**
- excellent: **mIoU ≥ 0.75**
- stretch: **mIoU ≥ 0.78**
- temporary-flood IoU: **≥ 0.55**


In [1]:
# ============================================================
# CELL: P6-C01 — ENVIRONMENT + REPRODUCIBILITY
# SINGLE-GPU SAFE: avoid nn.DataParallel CUDA issues
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # before importing torch

import sys
import gc
import json
import time
import math
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

torch.backends.cudnn.benchmark = True
AMP_ENABLED = torch.cuda.is_available()

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_V3_HighStandard")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"
ATTRIBUTION_ROOT = OUTPUT_ROOT / "attribution"

for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, CALIBRATION_ROOT, ATTRIBUTION_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 108)
print("P6-C01 — FLOODSHIFT-V2 HIGH-STANDARD MODEL [SINGLE-GPU SAFE]")
print("=" * 108)
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)

assert torch.cuda.device_count() <= 1


P6-C01 — FLOODSHIFT-V2 HIGH-STANDARD MODEL [SINGLE-GPU SAFE]
Python      : 3.12.13
PyTorch     : 2.10.0+cu128
Device      : cuda:0
Visible GPUs: 1
GPU         : Tesla T4
AMP         : True
Output root : /kaggle/working/FloodShift_V3_HighStandard


In [2]:
# ============================================================
# CELL: P6-C02 — LOCATE FROZEN NOTEBOOK-03 ARTIFACTS
# ============================================================

_file_cache = {}

def locate_by_saved_path(saved_path_or_name):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not locate '{filename}'. "
            "Attach Notebook 01 and the final Notebook 03 output."
        )

    parent_hint = saved.parent.name.lower()
    hinted = [
        p for p in matches
        if parent_hint and parent_hint in str(p).lower()
    ]

    candidates = hinted if hinted else matches

    selected = sorted(
        candidates,
        key=lambda p: (len(str(p)), str(p))
    )[0]

    _file_cache[cache_key] = selected
    return selected


FINAL_MANIFEST_PATH = locate_by_saved_path(
    "floodshift_preprocessing_manifest_v1.csv"
)

GLOBAL_LOCK_PATH = locate_by_saved_path(
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

OPTICAL_NORM_PATH = locate_by_saved_path(
    "optical_10band_train_normalization_exact.csv"
)

SAR_SCENE_STATS_PATH = locate_by_saved_path(
    "train_raw_sar_scene_statistics.csv"
)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 104)
print("P6-C02 — FROZEN PREPROCESSING FOUND")
print("=" * 104)

for p in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH
]:
    print(" ", p)

print("\nLock:", preprocessing_lock["status"])


# The V1 checkpoint is REQUIRED for controlled warm-starting.
V1_CHECKPOINT_MATCHES = list(
    INPUT_ROOT.rglob("FloodShift_Raw_Main_best.pt")
)

if not V1_CHECKPOINT_MATCHES:
    raise FileNotFoundError(
        "FloodShift_Raw_Main_best.pt was not found under /kaggle/input. "
        "Upload/attach the downloaded V1 checkpoint before continuing."
    )

V1_CHECKPOINT_PATH = sorted(
    V1_CHECKPOINT_MATCHES,
    key=lambda p: (len(str(p)), str(p))
)[0]

print("V1 warm-start checkpoint:", V1_CHECKPOINT_PATH)


P6-C02 — FROZEN PREPROCESSING FOUND
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_manifest_v1.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_GLOBAL_LOCK_v1.json
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/optical/optical_10band_train_normalization_exact.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/train_raw_sar_scene_statistics.csv

Lock: PERMANENTLY_FROZEN
V1 warm-start checkpoint: /kaggle/input/datasets/himavarshini2108/rawdata/FloodShift_Raw_Main_best.pt


In [3]:
# ============================================================
# CELL: P6-C03 — MANIFEST + TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = manifest[manifest["split"] == "train"].copy().reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].copy().reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].copy().reset_index(drop=True)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = np.isfinite(mu_all) & np.isfinite(sd_all) & (n_all > 0)

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std("vv_mean_db", "vv_std_db")
VH_MEAN, VH_STD = pooled_mean_std("vh_mean_db", "vh_std_db")

SAR_MEAN = np.array([VV_MEAN, VH_MEAN], dtype=np.float32)
SAR_STD = np.array([VV_STD, VH_STD], dtype=np.float32)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 104)
print("P6-C03 — DATA CONFIG")
print("=" * 104)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : renormalized P(clear)")
print("Main SAR branch    : Raw SAR")


P6-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : renormalized P(clear)
Main SAR branch    : Raw SAR


In [4]:
# ============================================================
# CELL: P6-C04 — TARGET + DATASET
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3
CLASS_NAMES = ["non_flood", "permanent_water", "temporary_flood"]

TILE_SIZE = 256
TILES_PER_SCENE = 4
TILE_OFFSETS = [(0,0), (0,256), (256,0), (256,256)]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(row["optical_reliability_path"]),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(flood.shape, IGNORE_INDEX, dtype=np.uint8)

    valid = (flood != -1)

    target[valid] = 0
    target[valid & (jrc == 1)] = 1
    target[valid & (jrc != 1) & (flood == 1)] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(np.float32, copy=False)

    valid = np.isfinite(sar).all(axis=0)
    out = np.zeros_like(sar, dtype=np.float32)

    for c in range(2):
        m = valid & np.isfinite(sar[c])
        out[c, m] = (sar[c, m] - SAR_MEAN[c]) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(np.float32, copy=False)

    valid = np.all(
        np.isfinite(optical_13) & (optical_13 != 0),
        axis=0
    )

    reflectance = optical_13 / 10000.0
    optical_10 = reflectance[OPTICAL_SOURCE_INDICES]

    out = (
        optical_10 - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(np.float32, copy=False)
    out[:, ~valid] = 0.0

    return out


def augment_all(sar, optical, reliability, target):
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(reliability, axis=1)
        target = np.flip(target, axis=1)

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(reliability, axis=0)
        target = np.flip(target, axis=0)

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(sar, k=k, axes=(1,2))
        optical = np.rot90(optical, k=k, axes=(1,2))
        reliability = np.rot90(reliability, k=k, axes=(0,1))
        target = np.rot90(target, k=k, axes=(0,1))

    return sar.copy(), optical.copy(), reliability.copy(), target.copy()


class FloodShiftDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.df = dataframe.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df) * TILES_PER_SCENE

    def __getitem__(self, index):
        scene_index = index // TILES_PER_SCENE
        tile_index = index % TILES_PER_SCENE

        row = self.df.iloc[scene_index]
        scene_id = str(row["scene_id"])

        y0, x0 = TILE_OFFSETS[tile_index]
        window = Window(x0, y0, TILE_SIZE, TILE_SIZE)

        paths = resolve_scene_files(row)

        with rasterio.open(paths["sar"]) as src:
            sar = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["optical"]) as src:
            optical_13 = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["flood"]) as src:
            flood = src.read(1, window=window)

        with rasterio.open(paths["jrc"]) as src:
            jrc = src.read(1, window=window)

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r"
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0+TILE_SIZE,
                x0:x0+TILE_SIZE
            ],
            dtype=np.float32
        )

        sar = normalize_sar(sar)
        optical = normalize_optical(optical_13)
        target = build_three_class_target(flood, jrc)

        reliability = np.clip(reliability, 0.0, 1.0)

        if self.train_mode:
            sar, optical, reliability, target = augment_all(
                sar, optical, reliability, target
            )

        return {
            "sar": torch.from_numpy(sar),
            "optical": torch.from_numpy(optical),
            "reliability": torch.from_numpy(reliability[None, ...]),
            "target": torch.from_numpy(target.astype(np.int64)),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }


In [5]:
# ============================================================
# CELL: P6-C05 — TARGET AUDIT + CLASS WEIGHTS
# ============================================================

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
ignored_count = 0

for position, (_, row) in enumerate(train_df.iterrows(), start=1):
    paths = resolve_scene_files(row)

    with rasterio.open(paths["flood"]) as src:
        flood = src.read(1)

    with rasterio.open(paths["jrc"]) as src:
        jrc = src.read(1)

    target = build_three_class_target(flood, jrc)

    for c in range(NUM_CLASSES):
        class_counts[c] += int((target == c).sum())

    ignored_count += int((target == IGNORE_INDEX).sum())

    if position == 1 or position % 75 == 0 or position == len(train_df):
        print(f"[{position:03d}/{len(train_df)}] target audit")

valid_total = class_counts.sum()
frequencies = class_counts / valid_total

# V3 restores the V1 inverse-frequency weighting because V2's
# softer sqrt-inverse weighting caused permanent-water IoU to collapse.
# This is frozen BEFORE any V3 test evaluation.
weights_np = 1.0 / np.maximum(frequencies, 1e-12)
weights_np = weights_np / weights_np.mean()

CLASS_WEIGHTS = torch.tensor(
    weights_np,
    dtype=torch.float32,
    device=DEVICE
)

target_audit = pd.DataFrame({
    "class_id": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "pixel_count": class_counts,
    "frequency": frequencies,
    "inverse_frequency_loss_weight": weights_np,
})

print("\n" + "=" * 104)
print("P6-C05 — TRAIN TARGET DISTRIBUTION")
print("=" * 104)
print(target_audit.to_string(index=False))
print("\nIgnored:", f"{ignored_count:,}")
print("Loss weights:", CLASS_WEIGHTS.detach().cpu().numpy())

target_audit.to_csv(
    METRIC_ROOT / "v3_train_target_distribution.csv",
    index=False
)

assert (class_counts > 0).all()


[001/314] target audit
[075/314] target audit
[150/314] target audit
[225/314] target audit
[300/314] target audit
[314/314] target audit

P6-C05 — TRAIN TARGET DISTRIBUTION
 class_id      class_name  pixel_count  frequency  inverse_frequency_loss_weight
        0       non_flood     64724902   0.892878                       0.063367
        1 permanent_water      1826059   0.025190                       2.246068
        2 temporary_flood      5939270   0.081932                       0.690565

Ignored: 9,822,985
Loss weights: [0.06336745 2.2460675  0.690565  ]


In [6]:
# ============================================================
# CELL: P6-C06 — DATALOADERS + ONE-BATCH CHECK
# ============================================================

BATCH_SIZE = 4
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_dataset = FloodShiftDataset(train_df, train_mode=True)
val_dataset = FloodShiftDataset(val_df, train_mode=False)
test_dataset = FloodShiftDataset(test_df, train_mode=False)

def make_loaders(seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)

    common = {
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "persistent_workers": (NUM_WORKERS > 0),
    }

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=g,
        **common
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()

batch = next(iter(train_loader))

print("=" * 104)
print("P6-C06 — MULTIMODAL BATCH CHECK")
print("=" * 104)

print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
print(
    "P(clear)   :",
    float(batch["reliability"].min()),
    "to",
    float(batch["reliability"].max())
)

assert batch["sar"].shape[1:] == (2,256,256)
assert batch["optical"].shape[1:] == (10,256,256)
assert batch["reliability"].shape[1:] == (1,256,256)
assert batch["target"].shape[1:] == (256,256)

assert torch.isfinite(batch["sar"]).all()
assert torch.isfinite(batch["optical"]).all()
assert torch.isfinite(batch["reliability"]).all()


P6-C06 — MULTIMODAL BATCH CHECK
SAR        : (4, 2, 256, 256)
Optical    : (4, 10, 256, 256)
Reliability: (4, 1, 256, 256)
Target     : (4, 256, 256)
P(clear)   : 0.000995635986328125 to 0.9970703125


## FloodShift-V3 architecture

```text
SAR encoder ─┐
             ├─ RIFT-Lite S3/S4 ─ reliability router ─┐
Opt encoder ─┘                                         │
P(clear) ──────────────────────────────────────────────┤
                                                      ▼
                                             Fusion Transformer
                                                      ▼
                                                   ASPP
                                                      ▼
                                     attention-gated decoder
                                      ├─ deep supervision
                                      ├─ boundary head
                                      └─ 3-class segmentation
```

`RIFT-Lite` is bounded local cross-attention rather than global all-to-all cross-attention. The optical-to-SAR direction is biased by `P(clear)`, while SAR assistance to the optical stream increases when optical reliability is low.


In [7]:
# ============================================================
# CELL: P6-C07 — FLOODSHIFT-V3 CONSERVATIVE ARCHITECTURE
#
# V2 diagnosis:
#   + temporary-flood validation IoU improved
#   - permanent-water validation IoU collapsed
#
# V3 therefore preserves the strong V1 backbone and introduces
# NEW capacity only as zero-initialized residual refinements.
#
# Changes:
#   1) RIFT only at S3, NOT S4
#   2) RIFT output projection starts at ZERO -> exact identity at start
#   3) ASPP residual starts at ZERO -> exact identity at start
#   4) low-weight deep supervision / boundary heads retained
# ============================================================

def make_group_norm(channels):
    for groups in [32, 16, 8, 4, 2, 1]:
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class BasicBlockSE(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = make_group_norm(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = make_group_norm(out_channels)
        self.se = SEBlock(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, 1, stride=stride, bias=False
                ),
                make_group_norm(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.act = nn.GELU()

    def forward(self, x):
        residual = self.shortcut(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        x = self.se(x)
        return self.act(x + residual)


class ModalityEncoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 5, stride=2, padding=2, bias=False),
            make_group_norm(64),
            nn.GELU(),
        )
        self.stage1 = self._make_stage(64, 64, blocks=2, stride=1)
        self.stage2 = self._make_stage(64, 128, blocks=2, stride=2)
        self.stage3 = self._make_stage(128, 256, blocks=2, stride=2)
        self.stage4 = self._make_stage(256, 512, blocks=2, stride=2)

    @staticmethod
    def _make_stage(in_channels, out_channels, blocks, stride):
        layers = [BasicBlockSE(in_channels, out_channels, stride=stride)]
        for _ in range(blocks - 1):
            layers.append(BasicBlockSE(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        s1 = self.stage1(x)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return [s1, s2, s3, s4]


class ReliabilityRouter(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = min(max(channels // 4, 16), 64)
        self.predictor = nn.Sequential(
            nn.Conv2d(2 * channels + 1, hidden, 1, bias=False),
            make_group_norm(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            make_group_norm(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, 2, 1),
        )

    def forward(self, sar_feature, optical_feature, p_clear):
        reliability = F.interpolate(
            p_clear,
            size=sar_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        router_input = torch.cat(
            [sar_feature, optical_feature, reliability], dim=1
        )
        logits = self.predictor(router_input)

        optical_prior = torch.clamp(reliability, min=1e-3, max=1.0)
        logits = torch.cat(
            [
                logits[:, 0:1],
                logits[:, 1:2] + torch.log(optical_prior),
            ],
            dim=1,
        )

        weights = torch.softmax(logits, dim=1)
        fused = (
            weights[:, 0:1] * sar_feature
            + weights[:, 1:2] * optical_feature
        )
        return fused, weights


class LocalCrossAttentionIdentity(nn.Module):
    """
    Local reliability-aware cross attention with an EXACT identity start.

    The final projection is initialized to zero. Therefore the downloaded
    V1 representation is not perturbed at initialization:
        output = query_feature + 0
    """
    def __init__(self, channels, kernel_size=3, reduction=8):
        super().__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.k2 = kernel_size * kernel_size
        self.hidden = max(channels // reduction, 16)

        self.q = nn.Conv2d(channels, self.hidden, 1, bias=False)
        self.k = nn.Conv2d(channels, self.hidden, 1, bias=False)
        self.v = nn.Conv2d(channels, channels, 1, bias=False)

        self.out = nn.Conv2d(channels, channels, 1, bias=False)
        nn.init.zeros_(self.out.weight)

        gate_hidden = max(channels // 8, 16)
        self.gate = nn.Sequential(
            nn.Conv2d(2 * channels + 1, gate_hidden, 1, bias=False),
            make_group_norm(gate_hidden),
            nn.GELU(),
            nn.Conv2d(gate_hidden, 1, 1),
            nn.Sigmoid(),
        )

        self.prior_strength = nn.Parameter(torch.tensor(0.5))

    def forward(self, query_feature, source_feature, source_reliability):
        b, c, h, w = query_feature.shape
        n = h * w
        pad = self.kernel_size // 2

        rel = F.interpolate(
            source_reliability,
            size=(h, w),
            mode="bilinear",
            align_corners=False,
        )

        q = self.q(query_feature).flatten(2)

        k = F.unfold(
            self.k(source_feature),
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, self.hidden, self.k2, n)

        scores = (
            q.unsqueeze(2) * k
        ).sum(dim=1) / math.sqrt(self.hidden)

        rel_windows = F.unfold(
            rel,
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, self.k2, n)

        scores = scores + self.prior_strength * torch.log(
            torch.clamp(rel_windows, min=1e-3, max=1.0)
        )

        attention = torch.softmax(scores, dim=1)

        v = F.unfold(
            self.v(source_feature),
            kernel_size=self.kernel_size,
            padding=pad,
        ).view(b, c, self.k2, n)

        context = (
            attention.unsqueeze(1) * v
        ).sum(dim=2).view(b, c, h, w)

        gate = self.gate(
            torch.cat([query_feature, source_feature, rel], dim=1)
        )

        return query_feature + gate * self.out(context)


class ConservativeRIFT(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.sar_from_optical = LocalCrossAttentionIdentity(channels)
        self.optical_from_sar = LocalCrossAttentionIdentity(channels)

    def forward(self, sar_feature, optical_feature, p_clear):
        sar_refined = self.sar_from_optical(
            sar_feature,
            optical_feature,
            p_clear,
        )

        sar_help_prior = 0.10 + 0.90 * (
            1.0 - torch.clamp(p_clear, 0.0, 1.0)
        )

        optical_refined = self.optical_from_sar(
            optical_feature,
            sar_feature,
            sar_help_prior,
        )

        return sar_refined, optical_refined


class FusionTransformer(nn.Module):
    def __init__(
        self,
        channels=512,
        token_count=256,
        heads=8,
        layers=2,
        dropout=0.1,
    ):
        super().__init__()
        self.channels = channels
        self.token_count = token_count
        self.position = nn.Parameter(torch.zeros(1, token_count, channels))

        layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=heads,
            dim_feedforward=channels * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(channels)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)

        if tokens.shape[1] != self.token_count:
            raise RuntimeError(
                f"Token count {tokens.shape[1]} != expected {self.token_count}"
            )

        tokens = self.norm(
            self.encoder(tokens + self.position)
        )
        return tokens.transpose(1, 2).reshape(b, c, h, w)


class IdentityASPP(nn.Module):
    """
    Multi-scale context with zero-initialized residual projection.
    At initialization this is exactly x -> x.
    """
    def __init__(self, channels=512, branch_channels=128):
        super().__init__()

        def branch(kernel, dilation=1):
            padding = 0 if kernel == 1 else dilation
            return nn.Sequential(
                nn.Conv2d(
                    channels,
                    branch_channels,
                    kernel,
                    padding=padding,
                    dilation=dilation,
                    bias=False,
                ),
                make_group_norm(branch_channels),
                nn.GELU(),
            )

        self.b1 = branch(1)
        self.b2 = branch(3, 2)
        self.b3 = branch(3, 4)

        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, branch_channels, 1, bias=False),
            make_group_norm(branch_channels),
            nn.GELU(),
        )

        self.project = nn.Conv2d(
            branch_channels * 4,
            channels,
            1,
            bias=False,
        )
        nn.init.zeros_(self.project.weight)

    def forward(self, x):
        pooled = self.pool(x)
        pooled = F.interpolate(
            pooled,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        residual = self.project(
            torch.cat(
                [self.b1(x), self.b2(x), self.b3(x), pooled],
                dim=1,
            )
        )

        return x + residual


class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels):
        super().__init__()
        inter_channels = max(skip_channels // 2, 16)

        self.gate = nn.Conv2d(
            gate_channels, inter_channels, 1, bias=False
        )
        self.skip = nn.Conv2d(
            skip_channels, inter_channels, 1, bias=False
        )

        self.psi = nn.Sequential(
            nn.GELU(),
            nn.Conv2d(inter_channels, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, gate_feature, skip_feature):
        attention = self.psi(
            self.gate(gate_feature) + self.skip(skip_feature)
        )
        return skip_feature * attention


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.attention = AttentionGate(
            in_channels,
            skip_channels,
        )

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels + skip_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            make_group_norm(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            make_group_norm(out_channels),
            nn.GELU(),
            SEBlock(out_channels),
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        skip = self.attention(x, skip)
        return self.conv(
            torch.cat([x, skip], dim=1)
        )


class FloodShiftV3(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.sar_encoder = ModalityEncoder(2)
        self.optical_encoder = ModalityEncoder(10)

        self.routers = nn.ModuleList(
            [
                ReliabilityRouter(64),
                ReliabilityRouter(128),
                ReliabilityRouter(256),
                ReliabilityRouter(512),
            ]
        )

        # Conservative: local cross-modal refinement only at S3.
        self.rift3 = ConservativeRIFT(256)

        self.transformer = FusionTransformer(
            channels=512,
            token_count=256,
            heads=8,
            layers=2,
            dropout=0.1,
        )

        self.context = IdentityASPP(
            channels=512,
            branch_channels=128,
        )

        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        self.final_decoder = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            make_group_norm(64),
            nn.GELU(),
            SEBlock(64),
        )

        self.head = nn.Conv2d(64, num_classes, 1)

        # Low-weight auxiliaries.
        self.aux_head2 = nn.Conv2d(128, num_classes, 1)

        self.boundary_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            make_group_norm(32),
            nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, sar, optical, p_clear, return_aux=False):
        sar_features = self.sar_encoder(sar)
        optical_features = self.optical_encoder(optical)

        sar_features[2], optical_features[2] = self.rift3(
            sar_features[2],
            optical_features[2],
            p_clear,
        )

        fused_features = []
        router_weights = []

        for sar_f, opt_f, router in zip(
            sar_features,
            optical_features,
            self.routers,
        ):
            fused, weights = router(
                sar_f,
                opt_f,
                p_clear,
            )
            fused_features.append(fused)
            router_weights.append(weights)

        x = self.transformer(
            fused_features[3]
        )
        x = self.context(x)

        d3 = self.decoder3(
            x,
            fused_features[2],
        )
        d2 = self.decoder2(
            d3,
            fused_features[1],
        )
        d1 = self.decoder1(
            d2,
            fused_features[0],
        )

        x = F.interpolate(
            d1,
            scale_factor=2.0,
            mode="bilinear",
            align_corners=False,
        )
        x = self.final_decoder(x)

        logits = self.head(x)

        if not return_aux:
            return logits

        aux2 = F.interpolate(
            self.aux_head2(d2),
            size=logits.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        boundary = self.boundary_head(x)

        return {
            "logits": logits,
            "aux2": aux2,
            "boundary": boundary,
            "router_weights": router_weights,
        }


In [8]:
# ============================================================
# CELL: P6-C08 — SHAPE CHECK + V1 WARM START
# ============================================================

model = FloodShiftV3(
    num_classes=NUM_CLASSES
).to(DEVICE)

with torch.no_grad():
    tiny = {
        "sar": batch["sar"][:1].to(DEVICE),
        "optical": batch["optical"][:1].to(DEVICE),
        "reliability": batch["reliability"][:1].to(DEVICE),
    }

    out = model(
        tiny["sar"],
        tiny["optical"],
        tiny["reliability"],
        return_aux=True,
    )

assert out["logits"].shape == (1, 3, 256, 256)
assert out["aux2"].shape == (1, 3, 256, 256)
assert out["boundary"].shape == (1, 1, 256, 256)

v1_checkpoint = torch.load(
    V1_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

v1_state = v1_checkpoint["state_dict"]

v1_state = {
    (key[7:] if key.startswith("module.") else key): value
    for key, value in v1_state.items()
}

current_state = model.state_dict()

compatible = {
    key: value
    for key, value in v1_state.items()
    if key in current_state
    and current_state[key].shape == value.shape
}

model.load_state_dict(
    compatible,
    strict=False,
)

loaded_params = sum(
    value.numel()
    for value in compatible.values()
)

total_params = sum(
    value.numel()
    for value in current_state.values()
)

warm_start_fraction = (
    loaded_params / total_params
)

print("=" * 112)
print("P6-C08 — V3 SHAPE + CONSERVATIVE V1 WARM START")
print("=" * 112)
print("V1 checkpoint epoch       :", v1_checkpoint.get("epoch"))
print(
    "V1 validation mIoU        :",
    v1_checkpoint.get("val_metrics", {}).get("mIoU")
)
print("Compatible tensors loaded :", len(compatible))
print(
    "Warm-start parameter frac :",
    f"{warm_start_fraction:.3%}"
)
print(
    "V3 parameters             :",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

assert warm_start_fraction > 0.75, (
    "Warm-start coverage too low. Stop before training."
)

del tiny, out

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


/tmp/ipykernel_58/3626552115.py:281: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=layers)


P6-C08 — V3 SHAPE + CONSERVATIVE V1 WARM START
V1 checkpoint epoch       : 7
V1 validation mIoU        : 0.5131610457622304
Compatible tensors loaded : 294
Warm-start parameter frac : 94.047%
V3 parameters             : 32,264,718


In [9]:
# ============================================================
# CELL: P6-C09 — CONSERVATIVE HIGH-STANDARD LOSS
#
# Goal:
#   preserve V1 permanent-water performance
#   while improving temporary-flood precision/IoU.
# ============================================================

LABEL_SMOOTHING = 0.01

# Main segmentation objective is deliberately close to V1.
CE_W = 0.45
DICE_W = 0.35
TEMP_TVERSKY_W = 0.20

AUX_W = 0.04
BOUNDARY_W = 0.03
MAIN_TOTAL_W = 1.0 - AUX_W - BOUNDARY_W

cross_entropy = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX,
    label_smoothing=LABEL_SMOOTHING,
)


def multiclass_dice_loss(logits, target, eps=1e-6):
    probabilities = torch.softmax(logits, dim=1)
    valid = target != IGNORE_INDEX

    safe_target = target.clone()
    safe_target[~valid] = 0

    one_hot = F.one_hot(
        safe_target,
        num_classes=NUM_CLASSES,
    ).permute(0, 3, 1, 2).float()

    valid_float = valid.unsqueeze(1).float()

    probabilities = probabilities * valid_float
    one_hot = one_hot * valid_float

    intersection = (
        probabilities * one_hot
    ).sum(dim=(0, 2, 3))

    denominator = (
        probabilities.sum(dim=(0, 2, 3))
        + one_hot.sum(dim=(0, 2, 3))
    )

    dice = (
        2.0 * intersection + eps
    ) / (
        denominator + eps
    )

    return 1.0 - dice.mean()


def temporary_flood_tversky_loss(
    logits,
    target,
    alpha=0.65,
    beta=0.35,
    gamma=1.25,
    eps=1e-6,
):
    """
    Temporary-flood-only focal Tversky.

    alpha > beta:
      false positives are penalized more than false negatives.

    This directly addresses V1's high recall / lower precision pattern
    without changing the permanent-water loss geometry.
    """
    probabilities = torch.softmax(
        logits,
        dim=1,
    )[:, 2]

    valid = (
        target != IGNORE_INDEX
    )

    target_temp = (
        target == 2
    ).float()

    p = probabilities[valid]
    y = target_temp[valid]

    tp = (p * y).sum()
    fp = (p * (1.0 - y)).sum()
    fn = ((1.0 - p) * y).sum()

    tversky = (
        tp + eps
    ) / (
        tp
        + alpha * fp
        + beta * fn
        + eps
    )

    return torch.pow(
        1.0 - tversky,
        gamma,
    )


def core_loss(logits, target):
    ce = cross_entropy(
        logits,
        target,
    )

    dice = multiclass_dice_loss(
        logits,
        target,
    )

    temp_tversky = temporary_flood_tversky_loss(
        logits,
        target,
    )

    total = (
        CE_W * ce
        + DICE_W * dice
        + TEMP_TVERSKY_W * temp_tversky
    )

    return total, ce, dice, temp_tversky


def make_boundary_target(target):
    valid = target != IGNORE_INDEX
    safe = target.clone()
    safe[~valid] = 0

    # Boundary supervision concentrates on temporary flood,
    # not every semantic transition.
    temp = (
        safe == 2
    )

    boundary = torch.zeros_like(
        temp,
        dtype=torch.bool,
    )

    h_valid = valid[:, :, 1:] & valid[:, :, :-1]
    h_edge = (
        temp[:, :, 1:] != temp[:, :, :-1]
    ) & h_valid

    boundary[:, :, 1:] |= h_edge
    boundary[:, :, :-1] |= h_edge

    v_valid = valid[:, 1:, :] & valid[:, :-1, :]
    v_edge = (
        temp[:, 1:, :] != temp[:, :-1, :]
    ) & v_valid

    boundary[:, 1:, :] |= v_edge
    boundary[:, :-1, :] |= v_edge

    return (
        boundary.unsqueeze(1).float(),
        valid.unsqueeze(1).float(),
    )


def boundary_loss(
    boundary_logits,
    target,
    eps=1e-6,
):
    boundary_target, valid = make_boundary_target(
        target
    )

    pos = (
        boundary_target * valid
    ).sum()

    neg = (
        (1.0 - boundary_target) * valid
    ).sum()

    pos_factor = torch.clamp(
        neg / torch.clamp(pos, min=1.0),
        min=1.0,
        max=12.0,
    )

    bce = F.binary_cross_entropy_with_logits(
        boundary_logits,
        boundary_target,
        reduction="none",
    )

    weights = (
        1.0
        + boundary_target * (pos_factor - 1.0)
    )

    bce = (
        bce * weights * valid
    ).sum() / torch.clamp(
        valid.sum(),
        min=1.0,
    )

    probability = torch.sigmoid(
        boundary_logits
    ) * valid

    target_v = (
        boundary_target * valid
    )

    intersection = (
        probability * target_v
    ).sum()

    dice = 1.0 - (
        2.0 * intersection + eps
    ) / (
        probability.sum()
        + target_v.sum()
        + eps
    )

    return 0.5 * bce + 0.5 * dice


def total_v3_loss(output, target):
    main, ce, dice, temp_tversky = core_loss(
        output["logits"],
        target,
    )

    aux, _, _, _ = core_loss(
        output["aux2"],
        target,
    )

    bnd = boundary_loss(
        output["boundary"],
        target,
    )

    total = (
        MAIN_TOTAL_W * main
        + AUX_W * aux
        + BOUNDARY_W * bnd
    )

    return {
        "total": total,
        "ce": ce.detach(),
        "dice": dice.detach(),
        "temp_tversky": temp_tversky.detach(),
        "aux": aux.detach(),
        "boundary": bnd.detach(),
    }


def confusion_from_logits(logits, target):
    prediction = torch.argmax(
        logits,
        dim=1,
    )

    valid = (
        target != IGNORE_INDEX
    )

    pred_v = prediction[
        valid
    ].detach().cpu()

    target_v = target[
        valid
    ].detach().cpu()

    encoded = (
        target_v * NUM_CLASSES
        + pred_v
    )

    cm = torch.bincount(
        encoded,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(
        NUM_CLASSES,
        NUM_CLASSES,
    )

    return cm.numpy().astype(
        np.int64
    )


def metrics_from_confusion(cm):
    cm = cm.astype(
        np.float64
    )

    tp = np.diag(cm)
    gt = cm.sum(axis=1)
    pred = cm.sum(axis=0)

    fp = pred - tp
    fn = gt - tp

    iou = tp / np.maximum(
        tp + fp + fn,
        1.0,
    )

    precision = tp / np.maximum(
        tp + fp,
        1.0,
    )

    recall = tp / np.maximum(
        tp + fn,
        1.0,
    )

    dice = (
        2.0 * tp
    ) / np.maximum(
        2.0 * tp + fp + fn,
        1.0,
    )

    accuracy = (
        tp.sum()
        / np.maximum(cm.sum(), 1.0)
    )

    result = {
        "mIoU": float(iou.mean()),
        "macro_dice": float(dice.mean()),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "accuracy": float(accuracy),
    }

    for index, name in enumerate(
        CLASS_NAMES
    ):
        result[
            f"iou_{name}"
        ] = float(iou[index])

        result[
            f"dice_{name}"
        ] = float(dice[index])

        result[
            f"precision_{name}"
        ] = float(precision[index])

        result[
            f"recall_{name}"
        ] = float(recall[index])

    return result


In [10]:
# ============================================================
# CELL: P6-C10 — STAGED FINE-TUNING + EMA
# ============================================================

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8

WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 2

# Preserve V1 knowledge with conservative base LRs.
LR_SAR_ENCODER = 1.0e-5
LR_OPT_ENCODER = 1.0e-5
LR_ROUTER = 2.0e-5
LR_TRANSFORMER = 2.0e-5
LR_DECODER = 3.0e-5
LR_HEAD = 4.0e-5

# New residual modules can learn faster.
LR_RIFT = 1.2e-4
LR_CONTEXT = 1.2e-4
LR_AUX = 1.0e-4

EMA_DECAY = 0.990

# Stage A protects the downloaded V1 model while new residual modules
# learn to become useful.
NEW_MODULE_WARMUP_EPOCHS = 2


class ModelEMA:
    def __init__(self, model, decay=0.990):
        self.decay = decay
        self.ema = copy.deepcopy(
            model
        ).eval()

        for parameter in self.ema.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        model_state = model.state_dict()
        ema_state = self.ema.state_dict()

        for key, ema_value in ema_state.items():
            model_value = model_state[
                key
            ].detach()

            if torch.is_floating_point(
                ema_value
            ):
                ema_value.mul_(
                    self.decay
                ).add_(
                    model_value,
                    alpha=1.0 - self.decay,
                )
            else:
                ema_value.copy_(
                    model_value
                )


def set_base_trainable(
    model,
    trainable,
):
    base_modules = [
        model.sar_encoder,
        model.optical_encoder,
        model.routers,
        model.transformer,
        model.decoder3,
        model.decoder2,
        model.decoder1,
        model.final_decoder,
        model.head,
    ]

    for module in base_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(
                trainable
            )

    # New V3 modules always remain trainable.
    for module in [
        model.rift3,
        model.context,
        model.aux_head2,
        model.boundary_head,
    ]:
        for parameter in module.parameters():
            parameter.requires_grad_(
                True
            )


def make_optimizer(model):
    groups = [
        {
            "params": model.sar_encoder.parameters(),
            "lr": LR_SAR_ENCODER,
            "name": "sar_encoder",
        },
        {
            "params": model.optical_encoder.parameters(),
            "lr": LR_OPT_ENCODER,
            "name": "optical_encoder",
        },
        {
            "params": model.routers.parameters(),
            "lr": LR_ROUTER,
            "name": "reliability_routers",
        },
        {
            "params": model.rift3.parameters(),
            "lr": LR_RIFT,
            "name": "conservative_rift_s3",
        },
        {
            "params": model.transformer.parameters(),
            "lr": LR_TRANSFORMER,
            "name": "fusion_transformer",
        },
        {
            "params": model.context.parameters(),
            "lr": LR_CONTEXT,
            "name": "identity_aspp",
        },
        {
            "params": (
                list(model.decoder3.parameters())
                + list(model.decoder2.parameters())
                + list(model.decoder1.parameters())
                + list(model.final_decoder.parameters())
            ),
            "lr": LR_DECODER,
            "name": "attention_decoder",
        },
        {
            "params": model.head.parameters(),
            "lr": LR_HEAD,
            "name": "main_head",
        },
        {
            "params": (
                list(model.aux_head2.parameters())
                + list(model.boundary_head.parameters())
            ),
            "lr": LR_AUX,
            "name": "auxiliary_heads",
        },
    ]

    return torch.optim.AdamW(
        groups,
        weight_decay=WEIGHT_DECAY,
    )


def run_epoch(
    model,
    loader,
    optimizer=None,
    scaler=None,
    ema=None,
):
    is_train = (
        optimizer is not None
    )

    model.train() if is_train else model.eval()

    running = {
        "loss": 0.0,
        "ce": 0.0,
        "dice": 0.0,
        "temp_tversky": 0.0,
        "aux": 0.0,
        "boundary": 0.0,
    }

    batches = 0

    confusion = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64,
    )

    router_sar_sum = np.zeros(
        4,
        dtype=np.float64,
    )

    router_opt_sum = np.zeros(
        4,
        dtype=np.float64,
    )

    router_count = np.zeros(
        4,
        dtype=np.float64,
    )

    if is_train:
        optimizer.zero_grad(
            set_to_none=True
        )

    for step, batch in enumerate(
        loader,
        start=1,
    ):
        sar = batch["sar"].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch["optical"].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch["reliability"].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.set_grad_enabled(
            is_train
        ):
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                output = model(
                    sar,
                    optical,
                    reliability,
                    return_aux=True,
                )

                losses = total_v3_loss(
                    output,
                    target,
                )

                backward_loss = (
                    losses["total"]
                    / GRAD_ACCUM_STEPS
                )

            if is_train:
                scaler.scale(
                    backward_loss
                ).backward()

                should_step = (
                    step % GRAD_ACCUM_STEPS == 0
                    or step == len(loader)
                )

                if should_step:
                    scaler.unscale_(
                        optimizer
                    )

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        GRAD_CLIP_NORM,
                    )

                    scaler.step(
                        optimizer
                    )
                    scaler.update()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    if ema is not None:
                        ema.update(
                            model
                        )

        running["loss"] += float(
            losses["total"].detach().cpu()
        )
        running["ce"] += float(
            losses["ce"].cpu()
        )
        running["dice"] += float(
            losses["dice"].cpu()
        )
        running["temp_tversky"] += float(
            losses["temp_tversky"].cpu()
        )
        running["aux"] += float(
            losses["aux"].cpu()
        )
        running["boundary"] += float(
            losses["boundary"].cpu()
        )

        batches += 1

        confusion += confusion_from_logits(
            output["logits"],
            target,
        )

        with torch.no_grad():
            for scale, weights in enumerate(
                output["router_weights"]
            ):
                router_sar_sum[
                    scale
                ] += float(
                    weights[:, 0].sum().detach().cpu()
                )

                router_opt_sum[
                    scale
                ] += float(
                    weights[:, 1].sum().detach().cpu()
                )

                router_count[
                    scale
                ] += float(
                    weights[:, 0].numel()
                )

    metrics = metrics_from_confusion(
        confusion
    )

    for key in running:
        output_key = (
            "loss"
            if key == "loss"
            else f"{key}_loss"
        )

        metrics[
            output_key
        ] = (
            running[key]
            / max(batches, 1)
        )

    for scale in range(4):
        denominator = max(
            router_count[scale],
            1.0,
        )

        metrics[
            f"router_sar_s{scale+1}"
        ] = (
            router_sar_sum[scale]
            / denominator
        )

        metrics[
            f"router_opt_s{scale+1}"
        ] = (
            router_opt_sum[scale]
            / denominator
        )

    return metrics


@torch.no_grad()
def evaluate_model(
    model,
    loader,
):
    return run_epoch(
        model=model,
        loader=loader,
        optimizer=None,
        scaler=None,
        ema=None,
    )


In [11]:
# ============================================================
# CELL: P6-C11 — TRAIN FLOODSHIFT-V3
#
# High-standard rule:
#   TEST is NEVER used for tuning or checkpoint selection.
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

train_loader, val_loader, test_loader = make_loaders(
    seed=SEED
)

model = model.to(
    DEVICE
)

# Stage A: freeze V1 base; train only zero-initialized residual modules.
set_base_trainable(
    model,
    trainable=False,
)

optimizer = make_optimizer(
    model
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=5e-7,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED,
)

ema = ModelEMA(
    model,
    decay=EMA_DECAY,
)

CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_V3_ConservativeRIFT_best.pt"
)

CHECKPOINT_EASY_PATH = Path(
    "/kaggle/working/FloodShift_V3_ConservativeRIFT_best.pt"
)

HISTORY_PATH = (
    HISTORY_ROOT
    / "FloodShift_V3_ConservativeRIFT_history.csv"
)

best_val_miou = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []

V1_VAL_MIOU = 0.5131610457622304
V1_VAL_TEMP_IOU = 0.3214
V1_VAL_PERM_IOU = 0.2630

# Higher promotion bar:
PROMOTION_VAL_MIOU = 0.540
PROMOTION_TEMP_IOU = 0.340
PROMOTION_PERM_IOU = 0.230

train_start = time.perf_counter()

print("=" * 120)
print("P6-C11 — TRAIN FLOODSHIFT-V3 [CONSERVATIVE RIFT]")
print("=" * 120)
print("Device               :", DEVICE)
print("Warm-start fraction  :", f"{warm_start_fraction:.3%}")
print("Stage-A frozen epochs:", NEW_MODULE_WARMUP_EPOCHS)
print("TEST USED HERE        : NO")

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):
    if (
        epoch
        ==
        NEW_MODULE_WARMUP_EPOCHS + 1
    ):
        set_base_trainable(
            model,
            trainable=True,
        )

        print(
            "\n>>> Stage B: V1 base unfrozen "
            "with conservative learning rates."
        )

    epoch_start = time.perf_counter()

    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scaler=scaler,
        ema=ema,
    )

    val_metrics = evaluate_model(
        ema.ema,
        val_loader,
    )

    scheduler.step(
        val_metrics["mIoU"]
    )

    learning_rates = {
        group.get(
            "name",
            f"group_{index}",
        ):
        group["lr"]

        for index, group in enumerate(
            optimizer.param_groups
        )
    }

    record = {
        "epoch": epoch,

        **{
            f"train_{key}": value
            for key, value
            in train_metrics.items()
        },

        **{
            f"val_{key}": value
            for key, value
            in val_metrics.items()
        },

        **{
            f"lr_{key}": value
            for key, value
            in learning_rates.items()
        },

        "epoch_sec":
            time.perf_counter()
            - epoch_start,
    }

    history.append(
        record
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss {train_metrics['loss']:.4f} | "
        f"EMA val mIoU {val_metrics['mIoU']:.4f} | "
        f"temp {val_metrics['iou_temporary_flood']:.4f} | "
        f"perm {val_metrics['iou_permanent_water']:.4f} | "
        f"temp-prec {val_metrics['precision_temporary_flood']:.4f} | "
        f"{record['epoch_sec']:.1f}s"
    )

    if (
        val_metrics["mIoU"]
        >
        best_val_miou + 1e-6
    ):
        best_val_miou = val_metrics[
            "mIoU"
        ]

        best_epoch = epoch
        epochs_without_improvement = 0

        payload = {
            "epoch":
                epoch,

            "state_dict":
                model.state_dict(),

            "ema_state_dict":
                ema.ema.state_dict(),

            "val_metrics":
                val_metrics,

            "warm_start_fraction":
                warm_start_fraction,

            "config": {
                "experiment":
                    "FloodShift_V3_ConservativeRIFT",

                "base_model":
                    "FloodShift-Raw V1",

                "test_used_for_selection":
                    False,

                "architecture": [
                    "zero-init residual RIFT at S3 only",
                    "zero-init residual ASPP",
                    "V1 router/transformer/decoder preserved",
                    "low-weight deep supervision",
                    "temporary-flood boundary head",
                ],

                "training": [
                    "2-epoch new-module warmup with V1 base frozen",
                    "V1 inverse-frequency class weights restored",
                    "temporary-flood-only asymmetric focal-Tversky",
                    "no modality dropout",
                    "EMA checkpoint selection",
                    "conservative differential LR",
                ],

                "differential_lr":
                    learning_rates,

                "ema_decay":
                    EMA_DECAY,

                "batch_size":
                    BATCH_SIZE,

                "gradient_accumulation":
                    GRAD_ACCUM_STEPS,
            },
        }

        torch.save(
            payload,
            CHECKPOINT_PATH,
        )

        torch.save(
            payload,
            CHECKPOINT_EASY_PATH,
        )

        print(
            "  ↳ Best EMA checkpoint updated"
        )

    else:
        epochs_without_improvement += 1

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch = {best_epoch}"
        )
        break

training_minutes = (
    time.perf_counter()
    - train_start
) / 60.0

assert CHECKPOINT_PATH.exists()
assert CHECKPOINT_EASY_PATH.exists()

best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_val = best_checkpoint[
    "val_metrics"
]

promotion_pass = (
    best_val["mIoU"]
    >=
    PROMOTION_VAL_MIOU
    and
    best_val["iou_temporary_flood"]
    >=
    PROMOTION_TEMP_IOU
    and
    best_val["iou_permanent_water"]
    >=
    PROMOTION_PERM_IOU
)

print("\n" + "=" * 120)
print("V3 TRAINING COMPLETE — VALIDATION-ONLY DECISION")
print("=" * 120)
print("Best epoch            :", best_epoch)
print("Best EMA val mIoU     :", f"{best_val['mIoU']:.4f}")
print("V1 val mIoU reference :", f"{V1_VAL_MIOU:.4f}")
print("Best val temp IoU     :", f"{best_val['iou_temporary_flood']:.4f}")
print("V1 val temp reference :", f"{V1_VAL_TEMP_IOU:.4f}")
print("Best val perm IoU     :", f"{best_val['iou_permanent_water']:.4f}")
print("V1 val perm reference :", f"{V1_VAL_PERM_IOU:.4f}")
print("Training minutes      :", f"{training_minutes:.2f}")
print("Checkpoint            :", CHECKPOINT_EASY_PATH)
print(
    "Promotion gate        :",
    "PASS ✅"
    if promotion_pass
    else "FAIL — DO NOT OPEN TEST YET"
)


P6-C11 — TRAIN FLOODSHIFT-V3 [CONSERVATIVE RIFT]
Device               : cuda:0
Warm-start fraction  : 94.047%
Stage-A frozen epochs: 2
TEST USED HERE        : NO
Epoch 01/30 | loss 0.6545 | EMA val mIoU 0.5146 | temp 0.3219 | perm 0.2672 | temp-prec 0.3665 | 75.7s
  ↳ Best EMA checkpoint updated
Epoch 02/30 | loss 0.6557 | EMA val mIoU 0.5447 | temp 0.3274 | perm 0.3582 | temp-prec 0.3701 | 45.7s
  ↳ Best EMA checkpoint updated

>>> Stage B: V1 base unfrozen with conservative learning rates.
Epoch 03/30 | loss 0.6060 | EMA val mIoU 0.4726 | temp 0.3000 | perm 0.1693 | temp-prec 0.3241 | 107.3s
Epoch 04/30 | loss 0.5597 | EMA val mIoU 0.4444 | temp 0.2917 | perm 0.0929 | temp-prec 0.3105 | 65.2s
Epoch 05/30 | loss 0.5346 | EMA val mIoU 0.4439 | temp 0.2910 | perm 0.0917 | temp-prec 0.3095 | 65.4s
Epoch 06/30 | loss 0.5173 | EMA val mIoU 0.4468 | temp 0.2960 | perm 0.0930 | temp-prec 0.3148 | 65.1s
Epoch 07/30 | loss 0.5084 | EMA val mIoU 0.4472 | temp 0.2959 | perm 0.0946 | temp-prec 0.

## Stop after C11 and inspect validation

Do **not** automatically open the test set.

If C11 prints `Promotion gate: PASS ✅`, the V3 configuration has passed the predeclared validation gate. Only then set `RUN_FINAL_TEST = True` in C12.

If it fails, keep the test locked. We will use the validation traces to design the next controlled change.


In [12]:
# ============================================================
# CELL: P6-C12 — FINAL TEST (LOCKED BY DEFAULT)
#
# Research rule:
#   Run C11 first.
#   If the validation promotion gate PASSes, change RUN_FINAL_TEST=True.
#   Do not tune V2 based on the resulting test number.
# ============================================================

RUN_FINAL_TEST = False

if not RUN_FINAL_TEST:
    print("=" * 112)
    print("FINAL TEST IS LOCKED 🔒")
    print("=" * 112)
    print("Best validation mIoU:", f"{best_val['mIoU']:.4f}")
    print(
        "Promotion gate:",
        "PASS ✅" if promotion_pass else "FAIL"
    )
    print(
        "\nIf PASS, set RUN_FINAL_TEST = True and rerun ONLY this cell."
    )
else:
    assert promotion_pass, (
        "Validation promotion gate failed. "
        "Do not use the test set to tune this configuration."
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    # Paper result uses the EMA weights selected by validation mIoU.
    model.load_state_dict(
        checkpoint["ema_state_dict"]
    )
    model.eval()

    val_metrics = evaluate_model(
        model,
        val_loader,
    )

    test_metrics = evaluate_model(
        model,
        test_loader,
    )

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    V1_TEST = {
        "model": "FloodShift-Raw V1",
        "test_mIoU": 0.663555,
        "test_iou_non_flood": 0.952405,
        "test_iou_permanent_water": 0.556947,
        "test_iou_temporary_flood": 0.481309,
        "test_macro_dice": 0.780301,
    }

    result_row = {
        "model": "FloodShift-V3-ConservativeRIFT",
        "parameters": parameter_count,
        "best_epoch": int(checkpoint["epoch"]),
        "training_minutes": float(training_minutes),
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }

    RESULT_PATH = (
        METRIC_ROOT
        / "FloodShift_V3_ConservativeRIFT_result.csv"
    )

    pd.DataFrame([result_row]).to_csv(
        RESULT_PATH,
        index=False,
    )

    comparison = pd.DataFrame(
        [
            V1_TEST,
            {
                "model": "FloodShift-V3-ConservativeRIFT",
                "test_mIoU": test_metrics["mIoU"],
                "test_iou_non_flood": test_metrics["iou_non_flood"],
                "test_iou_permanent_water":
                    test_metrics["iou_permanent_water"],
                "test_iou_temporary_flood":
                    test_metrics["iou_temporary_flood"],
                "test_macro_dice": test_metrics["macro_dice"],
            },
        ]
    )

    print("=" * 116)
    print("FLOODSHIFT-V3 FINAL TEST")
    print("=" * 116)
    print(
        comparison[
            [
                "model",
                "test_mIoU",
                "test_iou_permanent_water",
                "test_iou_temporary_flood",
                "test_macro_dice",
            ]
        ].to_string(index=False)
    )

    delta_miou = test_metrics["mIoU"] - V1_TEST["test_mIoU"]
    delta_temp = (
        test_metrics["iou_temporary_flood"]
        - V1_TEST["test_iou_temporary_flood"]
    )

    print("\nΔ mIoU vs V1      :", f"{delta_miou:+.4f}")
    print("Δ temp IoU vs V1  :", f"{delta_temp:+.4f}")

    print("\nAmbitious standards:")
    print("  Strong target   : mIoU >= 0.72")
    print("  Excellent target: mIoU >= 0.75")
    print("  Stretch target  : mIoU >= 0.78")
    print("  Temp-flood goal : IoU  >= 0.55")

    print("\nSaved:", RESULT_PATH)


FINAL TEST IS LOCKED 🔒
Best validation mIoU: 0.5447
Promotion gate: FAIL

If PASS, set RUN_FINAL_TEST = True and rerun ONLY this cell.


In [13]:
# ============================================================
# CELL: P6-C12A — PROTECT / DOWNLOAD V2 CHECKPOINT
# ============================================================

from IPython.display import FileLink, display
import hashlib
import shutil

source_checkpoint = CHECKPOINT_PATH
easy_checkpoint = Path(
    "/kaggle/working/FloodShift_V3_ConservativeRIFT_best.pt"
)

assert source_checkpoint.exists()

if source_checkpoint != easy_checkpoint:
    shutil.copy2(
        source_checkpoint,
        easy_checkpoint,
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("=" * 112)
print("P6-C12A — V2 CHECKPOINT PROTECTED ✅")
print("=" * 112)
print("Path  :", easy_checkpoint)
print("Size  :", f"{easy_checkpoint.stat().st_size/(1024**2):.2f} MB")
print("SHA256:", sha256_file(easy_checkpoint))

os.chdir("/kaggle/working")
display(
    FileLink(
        easy_checkpoint.name,
        result_html_prefix="⬇️ DOWNLOAD V2 CHECKPOINT: "
    )
)


P6-C12A — V2 CHECKPOINT PROTECTED ✅
Path  : /kaggle/working/FloodShift_V3_ConservativeRIFT_best.pt
Size  : 246.43 MB
SHA256: 0d645aa957aa83b0575e05e3bfc5ff85c93cb32cb3db667601fdc01762abf1ac


/kaggle/working/FloodShift_V3_ConservativeRIFT_best.pt

In [14]:
# ============================================================
# CELL: P6-C13 — OPTIONAL PAPER-GRADE SCENE BOOTSTRAP CI
#
# Run only AFTER C12 final test is legitimately unlocked.
# ============================================================

if "test_metrics" not in globals():
    print(
        "Final test has not been run. "
        "Bootstrap CI remains locked."
    )
else:
    @torch.no_grad()
    def scene_confusions(model, loader):
        by_scene = {}

        for batch in loader:
            sar = batch["sar"].to(DEVICE, non_blocking=True)
            optical = batch["optical"].to(DEVICE, non_blocking=True)
            reliability = batch["reliability"].to(DEVICE, non_blocking=True)
            target = batch["target"].to(DEVICE, non_blocking=True)

            logits = model(
                sar,
                optical,
                reliability,
            )

            for i, scene_id in enumerate(batch["scene_id"]):
                cm = confusion_from_logits(
                    logits[i:i+1],
                    target[i:i+1],
                )

                if scene_id not in by_scene:
                    by_scene[scene_id] = np.zeros(
                        (NUM_CLASSES, NUM_CLASSES),
                        dtype=np.int64,
                    )

                by_scene[scene_id] += cm

        return by_scene


    per_scene_cm = scene_confusions(
        model,
        test_loader,
    )

    scene_ids = sorted(per_scene_cm)
    scene_mious = []

    for scene_id in scene_ids:
        metrics = metrics_from_confusion(
            per_scene_cm[scene_id]
        )
        scene_mious.append(
            metrics["mIoU"]
        )

    rng = np.random.default_rng(SEED)
    bootstrap = []

    for _ in range(1000):
        sampled = rng.choice(
            scene_ids,
            size=len(scene_ids),
            replace=True,
        )

        cm = np.zeros(
            (NUM_CLASSES, NUM_CLASSES),
            dtype=np.int64,
        )

        for scene_id in sampled:
            cm += per_scene_cm[scene_id]

        bootstrap.append(
            metrics_from_confusion(cm)["mIoU"]
        )

    ci_low, ci_high = np.quantile(
        bootstrap,
        [0.025, 0.975],
    )

    CI_PATH = (
        METRIC_ROOT
        / "FloodShift_V3_test_scene_bootstrap_ci.json"
    )

    with open(CI_PATH, "w") as f:
        json.dump(
            {
                "n_test_scenes": len(scene_ids),
                "bootstrap_resamples": 1000,
                "test_mIoU": float(test_metrics["mIoU"]),
                "scene_bootstrap_mIoU_95CI": [
                    float(ci_low),
                    float(ci_high),
                ],
            },
            f,
            indent=2,
        )

    print("=" * 112)
    print("P6-C13 — SCENE-LEVEL BOOTSTRAP")
    print("=" * 112)
    print("Test scenes:", len(scene_ids))
    print("Test mIoU  :", f"{test_metrics['mIoU']:.4f}")
    print(
        "95% CI     :",
        f"[{ci_low:.4f}, {ci_high:.4f}]"
    )
    print("Saved      :", CI_PATH)


Final test has not been run. Bootstrap CI remains locked.


In [15]:
# ============================================================
# CELL: P6-C14 — SAVE V2 PROTOCOL
# ============================================================

protocol = {
    "model": "FloodShift-V3-ConservativeRIFT",
    "status": "validation-selected; test locked until promotion gate",
    "frozen_reference": {
        "FloodShift_Raw_V1_test_mIoU": 0.663555,
        "FloodShift_Raw_V1_test_temp_IoU": 0.481309,
        "FloodShift_Raw_V1_test_perm_IoU": 0.556947,
    },
    "ambitious_targets": {
        "strong_test_mIoU": 0.72,
        "excellent_test_mIoU": 0.75,
        "stretch_test_mIoU": 0.78,
        "temporary_flood_IoU": 0.55,
    },
    "data": {
        "retained_scenes": 440,
        "train": 314,
        "validation": 68,
        "test": 58,
        "tile_size": TILE_SIZE,
        "preprocessing": "Notebook 03 permanently frozen",
        "test_tuning_prohibited": True,
    },
    "architecture_changes_vs_v1": [
        "RIFT-Lite local reliability-biased cross-modal attention at S3/S4",
        "ASPP multi-scale context after Fusion Transformer",
        "deep supervision at D3/D2",
        "boundary auxiliary head",
    ],
    "training_changes_vs_v1": {
        "warm_start": "frozen V1 checkpoint, matching tensors only",
        "warm_start_fraction": float(warm_start_fraction),
        "class_weighting": "sqrt inverse frequency, mean normalized",
        "main_loss": "CE + Dice + asymmetric focal-Tversky",
        "auxiliary_losses": "deep supervision + boundary",
        "ema_decay": EMA_DECAY,
        "modality_dropout": {
            "optical": OPTICAL_MODALITY_DROPOUT,
            "sar": SAR_MODALITY_DROPOUT,
        },
        "gradient_accumulation": GRAD_ACCUM_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "selection_metric": "EMA validation mIoU",
    },
    "promotion_gate": {
        "validation_mIoU": PROMOTION_VAL_MIOU,
        "validation_temporary_flood_IoU": PROMOTION_TEMP_IOU,
    },
}

PROTOCOL_PATH = (
    OUTPUT_ROOT
    / "FloodShift_V3_protocol.json"
)

with open(PROTOCOL_PATH, "w") as f:
    json.dump(
        protocol,
        f,
        indent=2,
    )

print("=" * 112)
print("FLOODSHIFT-V3 NOTEBOOK PROTOCOL SAVED ✅")
print("=" * 112)
print("Protocol  :", PROTOCOL_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print(
    "\nResearch rule: never tune this V2 configuration "
    "using the final test score."
)


NameError: name 'OPTICAL_MODALITY_DROPOUT' is not defined